# C2.5 · Supply-chain research

**Function C — AI for Security Research → The Security Researcher**  ·  *Security of AI*

Builds on **[C2.4 · Data-layer research](https://spbreed.github.io/cyber-commons/lessons/C2.4.html)**.

| | |
|---|---|
| Open-source tooling | Sigstore, in-toto, OWASP AIBOM |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The hook

A model, a dataset, an adapter and an MCP server all arrive the way any dependency arrives — from someone else, usually unsigned, usually pinned to a tag that can move. Provenance questions produce real answers only when they are specific.

## 2 · The framework

```
   what arrives from someone else

   model weights   signed? by whom? pinned to a digest or a tag?
   dataset         provenance? licence? contaminated with your eval?
   adapter         who built it, against which base, verified how?
   MCP server      whose process? whose tool descriptions in your context?

   a moving tag is not a pin
```

Supply-chain research for AI systems is the ordinary software problem plus two
artefacts that have no mature process at all.

The ordinary part transfers directly: typosquatting, unsigned packages, new
packages with no soak time. The signals that predict a bad dependency have not
changed.

The two new artefacts:

- **Model weights.** Sigstore and in-toto attestation are technically possible
  and rare in practice. There is no download-count equivalent — "popular
  checkpoint" is not provenance, and a fine-tune of a fine-tune has a lineage
  nobody records.
- **Prompt and tool packages.** MCP servers, agent skill bundles, prompt
  libraries. These run *inside* your agent with your agent's authority, and
  there is no signing convention for them at all.

The honest output of this lesson includes stating where no answer currently
exists, because a risk assessment that invents one is worse than a gap.

## 3 · Demo — the ordinary signals still work

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Package:
    name: str; version: str; signed: bool = False
    downloads: int = 0; age_days: int = 999

KNOWN_GOOD = {"requests", "urllib3", "numpy", "pandas", "cryptography",
              "pytest", "flask", "colorama", "langchain"}

def levenshtein(a, b):
    if len(a) < len(b): a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]

def typosquat(pkg, known=KNOWN_GOOD):
    if pkg.name in known:
        return None
    near = sorted((levenshtein(pkg.name, k), k) for k in known)[:1]
    if near and near[0][0] <= 2:
        return f"distance {near[0][0]} from popular package {near[0][1]!r}"
    return None

def assess(pkg):
    flags = []
    if not pkg.signed:        flags.append("unsigned — no attestation to source")
    if pkg.age_days < 30:     flags.append(f"published {pkg.age_days}d ago — no soak time")
    if pkg.downloads < 1000:  flags.append(f"only {pkg.downloads} downloads")
    if (t := typosquat(pkg)): flags.append(t)
    verdict = "block" if len(flags) >= 3 else "review" if flags else "allow"
    return verdict, flags

for p in [Package("requests", "2.31.0", True, 900_000, 400),
          Package("requsts", "2.31.0", False, 12, 3),
          Package("colourama", "0.4.6", False, 40, 9),
          Package("langchain", "0.2.1", False, 400_000, 200)]:
    v, flags = assess(p)
    print(f"{p.name+'=='+p.version:24s}{v}")
    for f in flags: print(f"      · {f}")

## 4 · Where it breaks — the two artefacts with no process

In [ ]:
NEW_ARTEFACTS = {
 "model weights": {
   "signing": "Sigstore/in-toto possible, rarely used",
   "popularity signal": "NONE — 'popular checkpoint' is not provenance",
   "lineage": "a fine-tune of a fine-tune; base model often unrecorded",
   "runs with": "no authority of its own — but shapes every decision",
   "honest verdict": "assess the PUBLISHER, because you cannot assess the artefact"},
 "prompt / tool packages (MCP, skills)": {
   "signing": "NO convention exists",
   "popularity signal": "star counts, which are trivially gamed",
   "lineage": "none recorded",
   "runs with": "YOUR AGENT'S AUTHORITY — this is the dangerous one",
   "honest verdict": "treat as executable code, because it is"},
}
for artefact, props in NEW_ARTEFACTS.items():
    print(f"=== {artefact} ===")
    for k, v in props.items():
        print(f"   {k:20s} {v}")
    print()

In [ ]:
# An MCP tool package assessed with the ordinary signals — they still fire.
mcp_pkg = Package("mcp-jira-connector", "0.0.3", signed=False,
                  downloads=180, age_days=6)
v, flags = assess(mcp_pkg)
print(f"{mcp_pkg.name}: {v}")
for f in flags: print(f"   · {f}")
print("\nGood news: the existing process EXTENDS to it rather than needing")
print("invention. Bad news: nothing in that process accounts for the fact that")
print("this package will run with your agent's tools.")

def authority_weighted(pkg, runs_with_agent_authority, agent_blast):
    v, flags = assess(pkg)
    if runs_with_agent_authority and v != "allow":
        return "block", flags + [f"runs with agent authority (blast {agent_blast})"]
    return v, flags

v2, flags2 = authority_weighted(mcp_pkg, True, agent_blast=43)
print(f"\nauthority-weighted verdict: {v2}")
for f in flags2: print(f"   · {f}")
assert v2 == "block"

## 5 · The control — state the gap rather than inventing a number

In [ ]:
def risk_assessment(artefact, signals_available):
    known = [s for s, ok in signals_available.items() if ok]
    unknown = [s for s, ok in signals_available.items() if not ok]
    return {
      "artefact": artefact,
      "assessed_on": known,
      "cannot_assess": unknown,
      "statement": (f"assessed on {len(known)}/{len(signals_available)} signals; "
                    f"{', '.join(unknown)} not available for this artefact class"),
    }

for artefact, sig in (
  ("python package", {"signature": True, "downloads": True, "age": True, "lineage": True}),
  ("model weights",  {"signature": False, "downloads": False, "age": True, "lineage": False}),
  ("MCP tool pack",  {"signature": False, "downloads": False, "age": True, "lineage": False}),
):
    r = risk_assessment(artefact, sig)
    print(f"{r['artefact']:18s}{r['statement']}")
print("\nThat last sentence is the deliverable. A risk rating that hides which")
print("signals were unavailable is a number someone will later rely on.")

## What you just proved

The two legitimate packages are allowed or reviewed; both typosquats are blocked with the distance and the package they imitate. The MCP connector trips three ordinary signals and is escalated to block once agent authority is weighted in. The final assessments state explicitly which signals are unavailable for model weights and tool packages.

## Your turn

Add one question to your third-party assessment: "does this artefact execute with our agent's authority?" Anything answering yes should not be assessed on the same scale as a library.

---

**Next → [C2.6 · Benchmarks, reproducibility and the research harness](https://spbreed.github.io/cyber-commons/lessons/C2.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C2.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C2.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*